# Read recording metadata, one step at a time

This tutorial records your **live camera to S3 for 60 seconds**, then stops recording
so you can inspect the results at your own pace. Run the cells in order with
**Shift+Enter**. Each numbered step explains what the next cell does and what to look for.

You will read timestamps, duration, size, codec, motion decisions, storage updates,
and the complete metadata snapshot. Finally, you will save a small recording catalog
and acknowledge its events. Uploaded clips and local session files are retained.

## 1. Check the Python environment

From the repository root, run `uv sync`, then select this project's `.venv` Python
kernel in Jupyter or VS Code. The native recording dependencies must already be installed.
The notebook works when launched from the repository root or the `notebooks/` directory;
it imports the installed package without changing `sys.path`.

In [ ]:
"""Explore live HomeSight segment metadata and durable events."""

import json
import re
import sqlite3
import sys
import time
from collections.abc import Callable, Sequence
from contextlib import closing
from datetime import UTC, datetime
from html import escape
from pathlib import Path
from typing import cast
from uuid import uuid4

from IPython.display import HTML, JSON, clear_output, display
from pydantic import AnyHttpUrl, Field, SecretStr, ValidationError, field_validator
from pydantic_settings import BaseSettings, SettingsConfigDict

import homesight_core as core

# Contain IPython's untyped display callables at this boundary.
display_value = cast(Callable[[object], None], display)
html_value = cast(Callable[[str], object], HTML)
json_value = cast(Callable[[dict[str, object]], object], JSON)
clear_cell_output = cast(Callable[[bool], None], clear_output)

working_directory: Path = Path.cwd().resolve()
project_root: Path = (
    working_directory
    if (working_directory / "pyproject.toml").is_file()
    else working_directory.parent
)
if not (project_root / "src/homesight_core").is_dir():
    raise RuntimeError("Launch this notebook from the repository root or notebooks/.")

print("Python:", sys.executable)
print("HomeSight Core:", core.__version__)
print("Repository:", project_root)

## 2. Load your camera and S3 settings

Use the existing `.env` in the repository root. Required keys are `TEST_RTSP_URL`,
`TEST_S3_ENDPOINT`, `TEST_S3_BUCKET`, `TEST_S3_ACCESS_KEY`, and `TEST_S3_SECRET_KEY`.
`TEST_S3_SESSION_TOKEN` is optional. The S3 bucket must already exist.

Real environment variables override `.env`. Secret values stay on a typed settings
object; the output below only confirms that configuration loaded.

In [ ]:
class TutorialSettings(BaseSettings):
    """Load the existing live-camera and S3 configuration at the notebook boundary."""

    model_config = SettingsConfigDict(
        env_prefix="TEST_",
        env_file=project_root / ".env",
        env_file_encoding="utf-8",
        env_ignore_empty=True,
        extra="ignore",
        hide_input_in_errors=True,
    )

    rtsp_url: SecretStr = Field(min_length=1)
    s3_endpoint: AnyHttpUrl
    s3_bucket: str = Field(min_length=3)
    s3_access_key: SecretStr = Field(min_length=1)
    s3_secret_key: SecretStr = Field(min_length=1)
    s3_session_token: SecretStr | None = None

    @field_validator("rtsp_url")
    @classmethod
    def validate_rtsp_url(cls, value: SecretStr) -> SecretStr:
        """Require an RTSP scheme without exposing the credential-bearing URL."""
        if not value.get_secret_value().startswith(("rtsp://", "rtsps://")):
            raise ValueError("Use an rtsp:// or rtsps:// camera URL.")
        return value


try:
    settings = TutorialSettings()
except ValidationError as error:
    problems: list[str] = [
        f"TEST_{str(item['loc'][0]).upper()}: {item['msg']}"
        for item in error.errors(include_input=False, include_url=False)
    ]
    raise RuntimeError("Check your environment or .env settings:\n" + "\n".join(problems)) from None

print("Camera, endpoint, bucket, and credentials are configured.")
print("Optional session token:", "configured" if settings.s3_session_token else "not needed")

The next cell defines two display helpers. Tables use ordinary notebook HTML, so
you do not need pandas. Diagnostics redact credentials and camera URLs.

In [ ]:
def show_table(headers: Sequence[str], rows: Sequence[Sequence[object]]) -> None:
    """Display a small escaped table in Jupyter."""
    heading: str = "".join(f"<th>{escape(header)}</th>" for header in headers)
    body: str = "".join(
        "<tr>" + "".join(f"<td>{escape(str(value))}</td>" for value in row) + "</tr>"
        for row in rows
    )
    display_value(
        html_value(
            f"<div style='overflow-x:auto'><table><thead><tr>{heading}</tr></thead>"
            f"<tbody>{body}</tbody></table></div>"
        )
    )


def format_diagnostic(message: str | None, configuration: TutorialSettings) -> str:
    """Redact configured secrets and RTSP URLs before displaying a diagnostic."""
    result: str = message or "None"
    secrets: tuple[SecretStr | None, ...] = (
        configuration.rtsp_url,
        configuration.s3_access_key,
        configuration.s3_secret_key,
        configuration.s3_session_token,
    )
    for secret in secrets:
        if secret is not None and secret.get_secret_value():
            result = result.replace(secret.get_secret_value(), "[redacted]")
    return re.sub(r"rtsps?://[^\s<>]+", "[redacted RTSP URL]", result)

## 3. Configure a tutorial session

The next cell creates a fresh session directory and S3 prefix. It does not connect
to the camera yet. Edit `RECORD_SECONDS` or `SEGMENT_SECONDS` here if needed.
`TEST_CAMERA_TEST_SECONDS` belongs to the test suite; this tutorial uses its own duration.

Motion analysis keeps the default **30-second warm-up**. Early segments can therefore
have an unknown motion result. Clips without motion are retained. S3 uses the same
path-style addressing as the project's integration example, with TLS verification enabled.

Rerun this configuration cell for a **new** session. Rerun only the recording cell
to resume work in the **same** session and journal.

In [ ]:
RECORD_SECONDS = 60.0
SEGMENT_SECONDS = 10.0
S3_REGION = "us-east-1"

if RECORD_SECONDS <= 0 or SEGMENT_SECONDS <= 0:
    raise ValueError("Recording and segment durations must be positive.")

# A stopped recorder still owns its journal. Release it before replacing the session.
previous_recorder: object = globals().get("recorder")
if isinstance(previous_recorder, core.SegmentRecorder):
    previous_recorder.stop_recording()
previous_recorder = None

run_id: str = datetime.now(tz=UTC).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
session_dir: Path = project_root / "build/notebooks/example-trial" / run_id
session_dir.mkdir(parents=True, exist_ok=True)
catalog_path: Path = session_dir / "catalog.sqlite3"
s3_prefix: str = f"homesight-tutorial/{run_id}"

storage_config = core.S3StorageConfig(
    bucket_name=settings.s3_bucket,
    endpoint_url=str(settings.s3_endpoint).rstrip("/"),
    region=S3_REGION,
    key_prefix=s3_prefix,
    use_path_style_addressing=True,
    verify_tls_certificates=True,
    credentials=core.S3Credentials(
        access_key_id=settings.s3_access_key.get_secret_value(),
        secret_access_key=settings.s3_secret_key.get_secret_value(),
        session_token=(
            settings.s3_session_token.get_secret_value() if settings.s3_session_token else ""
        ),
    ),
)
config = core.SegmentRecorderConfig(
    camera=core.CameraStreamConfig(
        camera_name="example_trial",
        rtsp_url=settings.rtsp_url.get_secret_value(),
        rtsp_transport=core.RtspTransport.tcp,
    ),
    storage=storage_config,
    staging_directory=session_dir / "staging",
    segment_duration_seconds=SEGMENT_SECONDS,
    motion_analysis=core.MotionAnalysisParams(warmup_seconds=30),
    delete_segments_without_motion=False,
)
recorder: core.SegmentRecorder | None = None
events = list[core.RecorderEvent]()

show_table(
    ("Setting", "Value"),
    [
        ("Record for", f"{RECORD_SECONDS:g} seconds"),
        ("Target segment duration", f"{SEGMENT_SECONDS:g} seconds"),
        ("Motion warm-up", "30 seconds"),
        ("S3 destination", f"s3://{settings.s3_bucket}/{s3_prefix}/"),
        ("Local session", session_dir),
    ],
)

## 4. Record, collect events, and stop

**This cell starts live recording and uploads clips to S3.** It normally runs for
about 60 seconds plus shutdown time. A progress summary updates once a second.
Interrupting the cell still runs the stop logic and collects final available events.

Reading an event advances this recorder's delivery cursor; it does **not** acknowledge
the event. We deliberately keep this short session unacknowledged so you can inspect
it and demonstrate replay later. A continuously running application should persist
and acknowledge events as they arrive.

Stopping preserves unfinished analysis and uploads; it does not drain the entire
backlog. Pending outcomes below are real states, not successful uploads.

In [ ]:
if recorder is not None:
    recorder.stop_recording()
    recorder = None

events = []
recorder = core.SegmentRecorder(config)
started: float = time.monotonic()
deadline: float = started + RECORD_SECONDS
next_report: float = started

try:
    try:
        recorder.start_recording()
        while time.monotonic() < deadline:
            event: core.RecorderEvent | None = recorder.wait_for_next_recorder_event(timeout_ms=500)
            if event is not None:
                events.append(event)
            statistics: core.RecorderStatistics = recorder.get_recorder_statistics()
            now: float = time.monotonic()
            if now >= next_report:
                clear_cell_output(True)
                print(f"Recording: {min(now - started, RECORD_SECONDS):.0f}/{RECORD_SECONDS:g}s")
                print(f"State: {statistics.state.name}; events received: {len(events)}")
                print(
                    f"Segments written: {statistics.segments_written}; "
                    f"stored: {statistics.segments_stored}"
                )
                print(f"Waiting for upload: {statistics.segments_waiting_for_upload}")
                next_report = now + 1
            if statistics.state == core.RecorderState.failed:
                raise core.RecorderError(statistics.last_error or "Recorder entered failed state.")
    finally:
        try:
            recorder.stop_recording(finalize_timeout_ms=10000)
        finally:
            events.extend(recorder.take_recorder_events(max_count=100000))
            final_statistics: core.RecorderStatistics = recorder.get_recorder_statistics()
            clear_cell_output(True)
            show_table(
                ("Final statistic", "Value"),
                [
                    ("State", final_statistics.state.name),
                    ("Events received (still unacknowledged)", len(events)),
                    ("Segments written", final_statistics.segments_written),
                    ("Segments stored", final_statistics.segments_stored),
                    ("Waiting for upload", final_statistics.segments_waiting_for_upload),
                    ("Storage failures", final_statistics.storage_failures),
                    ("Reconnect attempts", final_statistics.reconnect_attempts),
                    ("Last diagnostic", format_diagnostic(final_statistics.last_error, settings)),
                ],
            )
except core.RecorderError as error:
    raise RuntimeError(format_diagnostic(str(error), settings)) from None

if final_statistics.state == core.RecorderState.failed:
    raise RuntimeError(format_diagnostic(final_statistics.last_error, settings))
if not events:
    print("No segments arrived. Check the camera connection and diagnostics, then rerun this cell.")

## 5. See the events and select a segment

Look for `segment_completed` followed by `storage_changed` for the same segment ID.
`sequence` orders events for this camera; `state_revision` orders changes to one segment.
The event ID is the identity used for acknowledgement.

In [ ]:
show_table(
    ("Sequence", "Event ID", "Kind", "Segment ID", "Revision", "Storage", "URI"),
    [
        (
            item.sequence,
            item.event_id,
            item.kind.name,
            item.completed_segment.segment.segment_id,
            item.completed_segment.state_revision,
            item.completed_segment.storage_outcome.name,
            item.completed_segment.storage_uri,
        )
        for item in sorted(events, key=lambda item: item.sequence)
    ],
)

Build one current snapshot per `(camera_name, segment_id)`, keeping the highest
revision. Select the first segment delivered in this session. If no events arrived,
the inspection cells explain that there is nothing to inspect instead of inventing data.

In [ ]:
latest_by_segment = dict[tuple[str, str], core.CompletedSegment]()
for item in sorted(events, key=lambda item: item.sequence):
    candidate: core.CompletedSegment = item.completed_segment
    key: tuple[str, str] = (candidate.segment.camera_name, candidate.segment.segment_id)
    previous: core.CompletedSegment | None = latest_by_segment.get(key)
    if previous is None or candidate.state_revision > previous.state_revision:
        latest_by_segment[key] = candidate

selected_key: tuple[str, str] | None = next(iter(latest_by_segment), None)
completed: core.CompletedSegment | None = (
    latest_by_segment[selected_key] if selected_key is not None else None
)
segment: core.SegmentInfo | None = completed.segment if completed is not None else None

if completed is not None and segment is not None:
    print("Camera:", segment.camera_name)
    print("Selected segment:", segment.segment_id)
    print("Latest revision:", completed.state_revision)
else:
    print("No segment to inspect. Resolve the connection issue and rerun step 4 onward.")

## 6. Read timestamps, duration, size, and codec

`start_epoch_ms` and `end_epoch_ms` are Unix milliseconds. Convert them to
timezone-aware UTC datetimes by dividing by 1,000. These are recorder-host times;
the end is derived from the finalized media duration.

In [ ]:
if segment is not None:
    start_time: datetime = datetime.fromtimestamp(segment.start_epoch_ms / 1000, tz=UTC)
    end_time: datetime = datetime.fromtimestamp(segment.end_epoch_ms / 1000, tz=UTC)
    print("Start milliseconds:", segment.start_epoch_ms)
    print("End milliseconds:", segment.end_epoch_ms)
    print("Start UTC:", start_time.isoformat())
    print("End UTC:", end_time.isoformat())

`duration_seconds` comes from the finalized container. Keyframe boundaries mean it
can differ from the configured split target. `file_size_bytes` is the original
finalized size, including when the local staging file has since been removed.
`video_codec` is `h264` or `h265`.

In [ ]:
if segment is not None:
    show_table(
        ("Field", "Value"),
        [
            ("duration_seconds", segment.duration_seconds),
            ("file_size_bytes", segment.file_size_bytes),
            ("video_codec", segment.video_codec),
            ("Original staging path", segment.staging_file_path),
            ("Staging file exists now", segment.staging_file_path.exists()),
        ],
    )

## 7. Interpret the motion result

There are two kinds of missing information:

| Value | Meaning |
| --- | --- |
| `completed.motion is None` | No analysis result is available, for example when analysis is disabled. |
| `motion.has_motion is None` | Analysis exists, but the decision is unknown. |
| `status == valid`, `has_motion is True` | Valid analysis detected motion. |
| `status == valid`, `has_motion is False` | Valid analysis found no motion. |

Warm-up, insufficient data, and failed analysis produce unknown decisions. A
segment touching the warm-up period remains unknown. The next cell lists actual
results from this run; a quiet scene may never produce a positive decision.

In [ ]:
def motion_decision(result: core.MotionAnalysisResult | None) -> str:
    """Keep unavailable and unknown decisions distinct from valid negative results."""
    if result is None:
        return "Unavailable"
    if result.status != core.MotionAnalysisStatus.valid or result.has_motion is None:
        return "Unknown"
    return "Motion" if result.has_motion else "No motion"


show_table(
    ("Segment ID", "Analysis status", "Decision"),
    [
        (
            value.segment.segment_id,
            value.motion.status.name if value.motion is not None else "None",
            motion_decision(value.motion),
        )
        for value in latest_by_segment.values()
    ],
)

Now inspect the selected segment's optional result. Changed-area metrics are
fractions between 0 and 1, displayed here as percentages only for a valid result.
Frame counts describe sampled frames, not every frame in the recording.

In [ ]:
motion: core.MotionAnalysisResult | None = completed.motion if completed is not None else None
if completed is not None:
    print("Motion decision:", motion_decision(motion))
    if motion is not None:
        print("Status:", motion.status.name)
        print("has_motion:", motion.has_motion)
        if motion.status == core.MotionAnalysisStatus.valid:
            print("Peak changed area:", f"{motion.peak_changed_area:.1%}")
            print("Mean changed area:", f"{motion.mean_changed_area:.1%}")
            print("Frames with motion:", motion.frames_with_motion)
            print("Frames analyzed:", motion.frames_analyzed)
        print("Motion diagnostic:", format_diagnostic(motion.error, settings))
    print("Analysis error:", format_diagnostic(completed.motion_analysis_error, settings))

## 8. Follow storage updates for this segment

The first completion can say `pending` with `storage_uri=None`. A later
`storage_changed` revision reports a storage success or failure. Inspect the outcome
and diagnostic together: a missing URI alone does not establish failure.

Use the storage URI to locate published footage. Successful S3 uploads release
their staging files by default. A quota event can remove local footage while
preserving a previously uploaded object's URI. All event kinds belong in the catalog.

In [ ]:
selected_events: list[core.RecorderEvent] = [
    item
    for item in sorted(events, key=lambda item: item.sequence)
    if (
        item.completed_segment.segment.camera_name,
        item.completed_segment.segment.segment_id,
    )
    == selected_key
]
show_table(
    ("Sequence", "Kind", "Revision", "Storage outcome", "Storage URI", "Diagnostic"),
    [
        (
            item.sequence,
            item.kind.name,
            item.completed_segment.state_revision,
            item.completed_segment.storage_outcome.name,
            item.completed_segment.storage_uri,
            format_diagnostic(item.completed_segment.storage_error, settings),
        )
        for item in selected_events
    ],
)

## 9. Inspect the complete snapshot and prepare JSON

`completed.to_dict()` includes the nested segment, optional motion result and its
parameters, storage fields, revision, and diagnostics. It preserves Python `Path`
objects and native enums. The first display shows those original Python types.

In [ ]:
if completed is not None:
    snapshot: dict[str, object] = completed.to_dict()
    display_value(snapshot)
    print("Path type:", type(completed.segment.staging_file_path).__name__)
    print("Storage enum type:", type(completed.storage_outcome).__name__)

For a JSON catalog, explicitly encode paths as strings and enums by name. The
converter below supports the native enum types present in a completed snapshot.
The stored JSON remains readable and retains `None` values as JSON `null`.

In [ ]:
def snapshot_json_default(value: object) -> str:
    """Encode the paths and native enums contained in CompletedSegment.to_dict()."""
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (core.SegmentStorageOutcome, core.MotionAnalysisStatus)):
        return value.name
    raise TypeError(f"Unsupported snapshot value type: {type(value).__name__}")


if completed is not None:
    serialized_snapshot: str = json.dumps(
        completed.to_dict(), default=snapshot_json_default, indent=2
    )
    json_snapshot: dict[str, object] = json.loads(serialized_snapshot)
    display_value(json_value(json_snapshot))

## 10. Observe replay before acknowledging

The recorder is stopped, but still owns the journal until its instance is released.
Release it and create a new instance with the same configuration. Do **not** start
recording: opening the journal is enough to read its unacknowledged events.

On the first pass, the IDs below match the events you already read in step 4.
If you rerun this cell after step 11, no replay is expected because those events
have been acknowledged. Keep this recorder alive for the acknowledgement step.

In [ ]:
if recorder is not None:
    recorder.stop_recording()
    recorder = None
recorder = core.SegmentRecorder(config)
replayed_events: list[core.RecorderEvent] = recorder.take_recorder_events(max_count=100000)
original_ids: set[str] = {item.event_id for item in events}
replayed_ids: set[str] = {item.event_id for item in replayed_events}

print("Previously read events:", len(original_ids))
print("Events replayed from the journal:", len(replayed_ids))
if replayed_ids:
    print("Replay includes every previously read event:", original_ids <= replayed_ids)
else:
    print("No unacknowledged events. They may already have been acknowledged in step 11.")

## 11. Commit a catalog update, then acknowledge its event

This small SQLite database stands in for your application's catalog. Its key is
`(camera_name, segment_id)`. An update replaces a row only when the incoming revision
is newer, so duplicate or older events cannot overwrite current state.

The persistence function returns **after its transaction commits**. Its caller then
acknowledges the event, including an already-applied revision. If persistence raises
an error, acknowledgement is never reached and the event remains replayable.

In [ ]:
with closing(sqlite3.connect(catalog_path)) as connection, connection:
    connection.execute(
        """
        CREATE TABLE IF NOT EXISTS segments (
            camera_name TEXT NOT NULL,
            segment_id TEXT NOT NULL,
            state_revision INTEGER NOT NULL,
            snapshot_json TEXT NOT NULL,
            PRIMARY KEY (camera_name, segment_id)
        )
        """
    )


def persist_catalog_update(item: core.RecorderEvent, database: Path) -> None:
    """Commit a segment revision, keeping a newer catalog snapshot if one exists."""
    value: core.CompletedSegment = item.completed_segment
    payload: str = json.dumps(value.to_dict(), default=snapshot_json_default)
    with closing(sqlite3.connect(database)) as catalog, catalog:
        catalog.execute(
            """
            INSERT INTO segments (camera_name, segment_id, state_revision, snapshot_json)
            VALUES (?, ?, ?, ?)
            ON CONFLICT (camera_name, segment_id) DO UPDATE SET
                state_revision = excluded.state_revision,
                snapshot_json = excluded.snapshot_json
            WHERE excluded.state_revision > segments.state_revision
            """,
            (
                value.segment.camera_name,
                value.segment.segment_id,
                value.state_revision,
                payload,
            ),
        )


print("Catalog ready:", catalog_path)

Run the next cell to apply the replayed events. Each output line is printed only
after the database transaction and acknowledgement have both returned successfully.
Repeating this cell is safe: the catalog update and acknowledgement are idempotent.

In [ ]:
if recorder is None:
    raise RuntimeError("Run step 10 first to open the session journal.")

for item in replayed_events:
    persist_catalog_update(item, catalog_path)
    recorder.acknowledge_recorder_events([item.event_id])
    print(
        f"Sequence {item.sequence}, revision {item.completed_segment.state_revision}: "
        "catalog committed, then event acknowledged."
    )

if not replayed_events:
    print("No replayed events to apply. The catalog is unchanged.")

Read back the catalog from disk. Several events for one segment result in one row
containing that segment's newest applied revision.

In [ ]:
with closing(sqlite3.connect(catalog_path)) as connection:
    catalog_rows: list[tuple[str, str, int, str]] = connection.execute(
        "SELECT camera_name, segment_id, state_revision, snapshot_json FROM segments ORDER BY rowid"
    ).fetchall()

show_table(
    ("Camera", "Segment ID", "Revision", "Stored outcome", "Stored URI"),
    [
        (
            camera,
            identity,
            revision,
            json.loads(payload)["storage_outcome"],
            json.loads(payload)["storage_uri"],
        )
        for camera, identity, revision, payload in catalog_rows
    ],
)
print("Persisted segment rows:", len(catalog_rows))

## 12. Verify acknowledgement survives reopening

Release the stopped instance and reopen the same journal once more. Previously
acknowledged events should no longer replay. This cell then releases the final
recorder so it does not retain the camera's state lock.

In [ ]:
if recorder is not None:
    recorder.stop_recording()
    recorder = None
recorder = core.SegmentRecorder(config)
try:
    remaining_events: list[core.RecorderEvent] = recorder.take_recorder_events(max_count=100000)
    print("Unacknowledged events after reopening:", len(remaining_events))
    assert not remaining_events, "Some events remain unacknowledged; repeat steps 10 and 11."
finally:
    recorder.stop_recording()
    recorder = None

print("Recorder released. Uploaded clips and local session files were retained.")
print("Local artifacts:", session_dir)
print("Catalog:", catalog_path)
print("S3 prefix:", f"s3://{settings.s3_bucket}/{s3_prefix}/")

## 13. Optional: sample up to N frames per second

This section uses one successfully uploaded clip from the session above.
It does **not** start another recording. Download a local copy first because
the sampling API accepts local file paths. The local copy and the S3 object
are retained; the original staging file may already have been released.

In [ ]:
from collections import Counter

import numpy as np
from numpy.typing import NDArray

uploaded_segments: list[core.CompletedSegment] = [
    value
    for value in latest_by_segment.values()
    if value.storage_outcome == core.SegmentStorageOutcome.stored and value.storage_uri is not None
]
sampling_metadata: core.CompletedSegment | None = (
    uploaded_segments[0] if uploaded_segments else None
)
sampling_path: Path | None = None
if sampling_metadata is not None:
    sampling_uri: str | None = sampling_metadata.storage_uri
    assert sampling_uri is not None
    sampling_directory: Path = session_dir / "sampling"
    sampling_directory.mkdir(parents=True, exist_ok=True)
    suffix: str = sampling_metadata.segment.staging_file_path.suffix or ".mp4"
    sampling_path = sampling_directory / (sampling_metadata.segment.segment_id + suffix)
    storage: core.SegmentStorage = core.create_segment_storage(storage_config)
    try:
        storage.download_segment_to_local_file(sampling_uri, sampling_path)
    except core.SegmentStorageError as error:
        raise RuntimeError(format_diagnostic(str(error), settings)) from None
    print("Downloaded segment:", sampling_metadata.segment.segment_id)
    print("Local sampling copy:", sampling_path)
else:
    print("No successful upload is available. Inspect the storage outcomes in step 8.")

Set a positive integer `FRAMES_PER_SECOND`. At 2, the sampler selects the
first real frame in each occupied interval `[0, 0.5)`, `[0.5, 1)`, and so on.
It includes the final partial second and leaves empty intervals empty.
Low-rate or irregular footage can therefore return fewer frames; it never
repeats a source frame to fill a gap.

The result contains owned RGB `uint8` pixels shaped `(count, height, width, 3)`
and one actual clip-relative timestamp per frame, truncated to milliseconds.
Results are limited to 4096 frames and 1 GiB of RGB pixels; exceeding a limit
raises an error rather than truncating the clip. The original fixed-count
sampler is still available when you want a specific total frame count.

In [ ]:
FRAMES_PER_SECOND = 2

sampled_frames: NDArray[np.uint8] | None = None
sampled_timestamps_ms = list[int]()
if sampling_path is not None:
    sampled_frames, sampled_timestamps_ms = core.sample_clip_frames_from_segment_at_rate(
        sampling_path,
        frames_per_second=FRAMES_PER_SECOND,
        frame_width=224,
        frame_height=224,
        decode_timeout_seconds=30,
    )
    print("RGB array shape:", sampled_frames.shape)
    print("RGB dtype:", sampled_frames.dtype)
    print("Actual clip-relative timestamps (ms):", sampled_timestamps_ms)
else:
    print("No local clip was downloaded, so sampling was skipped.")

Check the actual number of returned frames in each second. Adding a frame's
clip-relative offset to `start_epoch_ms` gives a recorder-host UTC estimate
you can associate with an embedding. This is derived from the host's segment
start, rather than a separate timestamp supplied by the camera.

These cells display metadata only. They do not run an embedding model or
write to Qdrant. The arrays are ready for downstream processing.

In [ ]:
if sampled_frames is not None and sampling_metadata is not None:
    per_second: Counter[int] = Counter(time_ms // 1000 for time_ms in sampled_timestamps_ms)
    show_table(("Clip second", "Frames returned"), sorted(per_second.items()))
    show_table(
        ("Frame index", "Clip offset (ms)", "Recorder-host UTC estimate"),
        [
            (
                index,
                offset,
                datetime.fromtimestamp(
                    (sampling_metadata.segment.start_epoch_ms + offset) / 1000, tz=UTC
                ).isoformat(),
            )
            for index, offset in enumerate(sampled_timestamps_ms)
        ],
    )
    assert len(sampled_timestamps_ms) == sampled_frames.shape[0]
    assert all(count <= FRAMES_PER_SECOND for count in per_second.values())

## Continue exploring

- **Inspect another segment:** change `selected_key` in step 5 to a key from
  `latest_by_segment`, update `completed` and `segment`, then rerun steps 6–9.
- **Resume this session:** rerun step 4 onward without rerunning step 3. The same
  staging directory and S3 configuration let unfinished jobs resume. New clips are recorded too.
- **Start a separate session:** rerun step 3, then step 4 onward. Earlier local
  files and S3 objects remain available under their original session paths.
- **Investigate pending or failed storage:** inspect the final statistics and storage
  event diagnostics. A successful notebook run does not imply every pending job finished.

Your SQLite catalog and the recorder's durable journal are separate databases.
Keep staging to preserve unfinished jobs; acknowledgements govern event replay,
not deletion of uploaded clips. This tutorial does not delete S3 objects.

See [Reading segment metadata](../docs/user-guide.md#reading-segment-metadata),
[recording and recovery](../docs/user-guide.md#recording-and-recovery), and the
[generated Python API](../build/docs/html/python-api.html) for the complete contracts.